# 01 - Setup et Chargement des Donnees BRFSS

**Projet** : Analyse de donnees medicales massives pour la prediction des MCV
**Dataset** : BRFSS 2020-2024 (>2M enregistrements)
**Environnement** : Kaggle Notebooks avec PySpark

In [140]:
# Installation des dependances necessaires
!pip install pyspark kagglehub imbalanced-learn xgboost -q

In [141]:
# Imports et configuration des graphiques
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Imports reussis")

Imports reussis


In [142]:
# Telechargement du dataset via kagglehub
import kagglehub

# Telechargement du dataset BRFSS 2020-2024
path = kagglehub.dataset_download("ajenks/brfss-2020-2024-cleaned-and-weighted")
print(f"Dataset telecharge dans : {path}")

# Lister les fichiers disponibles
for root, dirs, files in os.walk(path):
    for file in files:
        filepath = os.path.join(root, file)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"  {file} - {size_mb:.1f} MB")

Dataset telecharge dans : /kaggle/input/datasets/ajenks/brfss-2020-2024-cleaned-and-weighted
  brfss_2021_eda.parquet - 34.7 MB
  brfss_2021_ml.parquet - 29.6 MB
  brfss_2024_ml.parquet - 29.2 MB
  brfss_2023_eda.parquet - 38.4 MB
  brfss_2022_ml.parquet - 29.4 MB
  brfss_2023_ml.parquet - 33.5 MB
  brfss_2022_eda.parquet - 34.5 MB
  brfss_2020_2024_pooled_eda.parquet - 119.9 MB
  brfss_2020_eda.parquet - 28.3 MB
  brfss_2020_2024_pooled_ml.parquet - 79.5 MB
  brfss_2024_eda.parquet - 34.5 MB
  brfss_2020_ml.parquet - 23.8 MB


In [143]:
# Initialisation de SparkSession en mode local
spark = SparkSession.builder \
    .appName("BRFSS_CVD_Prediction") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# Verification
print(f"Spark version : {spark.version}")
print(f"Spark UI : {spark.sparkContext.uiWebUrl}")

Spark version : 4.0.2
Spark UI : http://ebb29f1b77fc:4040


26/05/28 22:59:56 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [144]:
# Chargement du Parquet avec Spark
parquet_files = [f for f in os.listdir(path) if f.endswith(".parquet")]
print(f"Fichiers Parquet trouves : {parquet_files}")

if not parquet_files:
    raise FileNotFoundError("Aucun fichier .parquet trouve dans le dossier Kaggle.")

# Priorite au fichier pooled ML si disponible
preferred = "brfss_2020_2024_pooled_ml.parquet"
parquet_path = os.path.join(path, preferred if preferred in parquet_files else sorted(parquet_files)[0])

print(f"Fichier selectionne : {parquet_path}")
df_spark = spark.read.parquet(parquet_path).cache()

row_count = df_spark.count()
col_count = len(df_spark.columns)
print(f"Dataset charge : {row_count:,} lignes x {col_count} colonnes")


Fichiers Parquet trouves : ['brfss_2021_eda.parquet', 'brfss_2021_ml.parquet', 'brfss_2024_ml.parquet', 'brfss_2023_eda.parquet', 'brfss_2022_ml.parquet', 'brfss_2023_ml.parquet', 'brfss_2022_eda.parquet', 'brfss_2020_2024_pooled_eda.parquet', 'brfss_2020_eda.parquet', 'brfss_2020_2024_pooled_ml.parquet', 'brfss_2024_eda.parquet', 'brfss_2020_ml.parquet']
Fichier selectionne : /kaggle/input/datasets/ajenks/brfss-2020-2024-cleaned-and-weighted/brfss_2020_2024_pooled_ml.parquet
Dataset charge : 2,176,776 lignes x 129 colonnes


26/05/28 22:59:57 WARN CacheManager: Asked to cache already cached data.


In [145]:
# Inspection du schema
df_spark.printSchema()

root
 |-- ACEDEPRS: double (nullable = true)
 |-- ACEDIVRC: double (nullable = true)
 |-- ACEDRINK: double (nullable = true)
 |-- ACEDRUGS: double (nullable = true)
 |-- ACEHURT1: double (nullable = true)
 |-- ACEHVSEX: double (nullable = true)
 |-- ACEPRISN: double (nullable = true)
 |-- ACEPUNCH: double (nullable = true)
 |-- ACESWEAR: double (nullable = true)
 |-- ACETOUCH: double (nullable = true)
 |-- ACETTHEM: double (nullable = true)
 |-- ADDEPEV3: double (nullable = true)
 |-- ASTHMA3: double (nullable = true)
 |-- ASTHNOW: double (nullable = true)
 |-- BLIND: double (nullable = true)
 |-- CAREGIV1: double (nullable = true)
 |-- CASTHDX2: double (nullable = true)
 |-- CASTHNO2: double (nullable = true)
 |-- CHCKDNY2: double (nullable = true)
 |-- CHECKUP1: double (nullable = true)
 |-- CHILDREN: double (nullable = true)
 |-- CHKHEMO3: double (nullable = true)
 |-- CNCRAGE: double (nullable = true)
 |-- CNCRDIFF: double (nullable = true)
 |-- CRGVALZD: double (nullable = true)
 

In [146]:
# Apercu des donnees
df_spark.show(5, truncate=False)

+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+-------+-------+-----+--------+--------+--------+--------+--------+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+-------+--------+--------+--------+--------+----+------+--------+--------+--------+--------+--------+-----+-------+-----------+--------+--------+--------+--------+-------+-----------+-----------+--------+------+--------------+------------+------------+----------------+-------------------+----------+-------+-------+--------+-------+--------+-----+-----+--------+--------+--------+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+--------+--------------+--------+------+--------+--------+--------+--------+------+----------+--------+-----------+--------+-------+--------+-------+------+--------+------+--------+------+--------+--------+------+--------+------------+---

In [147]:
# Statistiques descriptives
df_spark.describe().show()

26/05/28 23:01:30 WARN DAGScheduler: Broadcasting large task binary with size 1602.3 KiB


+-------+-------------------+------------------+------------------+------------------+------------------+-------------------+-------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+-------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+-------------------+-------------------+------------------+-------------------+------------------+-------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+--------

In [148]:
# Verification de la variable cible
candidates = []
if "SELECTED_COLUMNS" in globals():
    candidates.append(SELECTED_COLUMNS[0])
candidates += ["HeartDiseaseorAttack", "HAS_HEART_ATTACK", "CVDCRHD4", "_MICHD"]

target_col = next((c for c in candidates if c in df_spark.columns), None)
if target_col is None:
    raise ValueError("Aucune colonne cible trouvee dans df_spark.")

print(f"Distribution de {target_col} :")
total = row_count if "row_count" in globals() else df_spark.count()
(df_spark.groupBy(target_col).count()
    .withColumn("pourcentage", F.round(F.col("count") / F.lit(total) * 100, 2))
    .orderBy(target_col)
    .show())

Distribution de HAS_HEART_ATTACK :
+----------------+-------+-----------+
|HAS_HEART_ATTACK|  count|pourcentage|
+----------------+-------+-----------+
|            NULL|  13486|       0.62|
|             0.0|2043271|      93.87|
|             1.0| 120019|       5.51|
+----------------+-------+-----------+



In [149]:
# Colonnes du dataset
print(f"{len(df_spark.columns)} colonnes :")
for i, col in enumerate(df_spark.columns, 1):
    print(f"  {i:2d}. {col}")

129 colonnes :
   1. ACEDEPRS
   2. ACEDIVRC
   3. ACEDRINK
   4. ACEDRUGS
   5. ACEHURT1
   6. ACEHVSEX
   7. ACEPRISN
   8. ACEPUNCH
   9. ACESWEAR
  10. ACETOUCH
  11. ACETTHEM
  12. ADDEPEV3
  13. ASTHMA3
  14. ASTHNOW
  15. BLIND
  16. CAREGIV1
  17. CASTHDX2
  18. CASTHNO2
  19. CHCKDNY2
  20. CHECKUP1
  21. CHILDREN
  22. CHKHEMO3
  23. CNCRAGE
  24. CNCRDIFF
  25. CRGVALZD
  26. CSRVCLIN
  27. CSRVCTL2
  28. CSRVDEIN
  29. CSRVDOC1
  30. CSRVINSR
  31. CSRVINST
  32. CSRVPAIN
  33. CSRVRTRN
  34. CSRVSUM
  35. CSRVTRT3
  36. CVDCRHD4
  37. CVDINFR4
  38. CVDSTRK3
  39. DEAF
  40. DECIDE
  41. DIABETE4
  42. DIFFALON
  43. DIFFDRES
  44. DIFFWALK
  45. DRNK3GE5
  46. EDUCA
  47. EMPLOY1
  48. EVER_SMOKED
  49. EXERANY2
  50. EYEEXAM1
  51. FLSHTMY3
  52. FLUSHOT7
  53. GENHLTH
  54. GOOD_HEALTH
  55. GOT_FLUSHOT
  56. HADHYST2
  57. HADMAM
  58. HAS_DEPRESSION
  59. HAS_DIABETES
  60. HAS_EXERCISE
  61. HAS_HEART_ATTACK
  62. HAS_PERSONAL_DOCTOR
  63. HAS_STROKE
  64. HEIGHT3
  

In [150]:
# Binarisation BRFSS + Selection des variables standardisees

import re

# ============================================================
# STEP 1 : Binarisation des codes BRFSS bruts en 0/1
# Les variables BRFSS utilisent le codage : 1=Oui, 2=Non,
# 7=Ne sait pas, 9=Refuse, 88=Aucun (pour les jours).
# On convertit tout en 0.0/1.0 avant la selection.
# ============================================================

# HeartDiseaseorAttack : _MICHD = maladie coronarienne OU infarctus (~9.4%)
# _MICHD : 1=Oui (CVDCRHD4==1 OU CVDINFR4==1), 2=Non
if "_MICHD" in df_spark.columns:
    df_spark = df_spark.withColumn(
        "HeartDiseaseorAttack",
        F.when(F.col("_MICHD") == 1, 1.0)
         .when(F.col("_MICHD") == 2, 0.0)
         .otherwise(None)
    )

# DiffWalk : DIFFWALK 1=Oui→1.0, 2=Non→0.0
if "DIFFWALK" in df_spark.columns:
    df_spark = df_spark.withColumn(
        "DiffWalk",
        F.when(F.col("DIFFWALK") == 1, 1.0)
         .when(F.col("DIFFWALK") == 2, 0.0)
         .otherwise(None)
    )

# Sex : 1=Homme→0.0, 2=Femme→1.0  (convention ML : 0=reference)
sex_src = next((c for c in ["_SEX", "SEXVAR"] if c in df_spark.columns), None)
if sex_src:
    df_spark = df_spark.withColumn(
        "Sex",
        F.when(F.col(sex_src) == 1, 0.0)
         .when(F.col(sex_src) == 2, 1.0)
         .otherwise(None)
    )

# MentHlth : 88="Aucun jour"→0, 0-30=valide, 77/99→null
if "MENTHLTH" in df_spark.columns:
    df_spark = df_spark.withColumn(
        "MentHlth",
        F.when(F.col("MENTHLTH") == 88, 0.0)
         .when(F.col("MENTHLTH").between(0, 30), F.col("MENTHLTH").cast("double"))
         .otherwise(None)
    )

# PhysHlth : meme logique que MentHlth
if "PHYSHLTH" in df_spark.columns:
    df_spark = df_spark.withColumn(
        "PhysHlth",
        F.when(F.col("PHYSHLTH") == 88, 0.0)
         .when(F.col("PHYSHLTH").between(0, 30), F.col("PHYSHLTH").cast("double"))
         .otherwise(None)
    )

# GenHlth : GENHLTH 1-5 valide (1=Excellent…5=Mauvaise), 7/9→null
if "GENHLTH" in df_spark.columns:
    df_spark = df_spark.withColumn(
        "GenHlth",
        F.when(F.col("GENHLTH").between(1, 5), F.col("GENHLTH").cast("double"))
         .otherwise(None)
    )

# Age : _AGEG5YR 1-13 valide (tranches de 5 ans), 14→null
if "_AGEG5YR" in df_spark.columns:
    df_spark = df_spark.withColumn(
        "Age",
        F.when(F.col("_AGEG5YR").between(1, 13), F.col("_AGEG5YR").cast("double"))
         .otherwise(None)
    )

# Education : EDUCA 1-6 valide, 9→null
if "EDUCA" in df_spark.columns:
    df_spark = df_spark.withColumn(
        "Education",
        F.when(F.col("EDUCA").between(1, 6), F.col("EDUCA").cast("double"))
         .otherwise(None)
    )

# BMI : _BMI5 est stocke en centiemes (ex: 2800 = 28.00 kg/m²)
if "BMI" not in df_spark.columns:
    if "_BMI5" in df_spark.columns:
        df_spark = df_spark.withColumn("BMI", F.col("_BMI5") / 100.0)
    elif "_BMI5_SCALED" in df_spark.columns:
        df_spark = df_spark.withColumn("BMI", F.col("_BMI5_SCALED"))
if "BMI" in df_spark.columns:
    df_spark = df_spark.withColumn(
        "BMI", F.when(F.col("BMI").between(10.0, 99.0), F.col("BMI")).otherwise(None)
    )

# HvyAlcoholConsump : DRNK3GE5 = nb de jours de binge drinking (0-30)
# -> binaire : >0 = consommation excessive = 1, 0 = pas de binge = 0
if "DRNK3GE5" in df_spark.columns:
    df_spark = df_spark.withColumn(
        "HvyAlcoholConsump",
        F.when(F.col("DRNK3GE5") > 0, 1.0)
         .when(F.col("DRNK3GE5") == 0, 0.0)
         .otherwise(None)
    )

print("Binarisation BRFSS appliquee")

# ============================================================
# STEP 2 : Selection des colonnes avec preference pour
# colonnes deja derivees en binaire (HAS_STROKE, etc.)
# ============================================================

REQUESTED_COLUMNS = [
    'HeartDiseaseorAttack', 'HighBP', 'HighChol', 'BMI', 'Smoker', 'Stroke',
    'Diabetes', 'PhysActivity', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk',
    'Sex', 'Age', 'Education', 'Income', 'HvyAlcoholConsump', 'NoDocbcCost'
]

col_candidates = {
    "HeartDiseaseorAttack": ["HeartDiseaseorAttack", "HAS_HEART_ATTACK", "_MICHD"],
    "HighBP":               ["HighBP", "BPHIGH4"],
    "HighChol":             ["HighChol", "TOLDHI2"],
    "BMI":                  ["BMI", "_BMI5_SCALED"],
    # EVER_SMOKED et HAS_STROKE/DIABETES/EXERCISE sont des flags binaires derives
    "Smoker":               ["EVER_SMOKED", "_RFSMOK3", "SMOKE100", "_SMOKER3"],
    "Stroke":               ["HAS_STROKE", "Stroke", "CVDSTRK3"],
    "Diabetes":             ["HAS_DIABETES", "Diabetes", "DIABETE4"],
    "PhysActivity":         ["HAS_EXERCISE", "PhysActivity", "EXERANY2"],
    "GenHlth":              ["GenHlth", "GENHLTH"],
    "MentHlth":             ["MentHlth", "MENTHLTH"],
    "PhysHlth":             ["PhysHlth", "PHYSHLTH"],
    "DiffWalk":             ["DiffWalk", "DIFFWALK"],
    "Sex":                  ["Sex", "_SEX", "SEXVAR"],
    "Age":                  ["Age", "_AGEG5YR"],
    "Education":            ["Education", "EDUCA"],
    "Income":               ["Income", "INCOME2", "_INCOMG"],
    "HvyAlcoholConsump":    ["HvyAlcoholConsump", "DRNK3GE5"],
    "NoDocbcCost":          ["NoDocbcCost", "MEDCOST"]
}

def norm(name):
    return re.sub(r'[^a-z0-9]+', '', name.lower())

norm_map = {norm(c): c for c in df_spark.columns}

def pick_col(cands):
    for c in cands:
        if c in df_spark.columns:
            return c
    for c in cands:
        if norm(c) in norm_map:
            return norm_map[norm(c)]
    return None

resolved = {name: pick_col(col_candidates.get(name, [name])) for name in REQUESTED_COLUMNS}
missing  = [n for n, src in resolved.items() if src is None]
available = [n for n, src in resolved.items() if src is not None]

if missing:
    print(f"Colonnes absentes du fichier _ml pooled (non disponibles) : {missing}")

df_spark = df_spark.select([F.col(resolved[n]).alias(n) for n in available])
print(f"\n{len(available)}/{len(REQUESTED_COLUMNS)} colonnes selectionnees : {available}")

# Verification de la prevalence cible
if "HeartDiseaseorAttack" in available:
    total = df_spark.count()
    pos   = df_spark.filter(F.col("HeartDiseaseorAttack") == 1.0).count()
    print(f"\nPrevalence MCV : {pos/total*100:.2f}% (cible PRD ~9.4%)")

df_spark.show(5)

Binarisation BRFSS appliquee
Colonnes absentes du fichier _ml pooled (non disponibles) : ['HighBP', 'HighChol', 'Income', 'NoDocbcCost']

14/18 colonnes selectionnees : ['HeartDiseaseorAttack', 'BMI', 'Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'HvyAlcoholConsump']

Prevalence MCV : 8.63% (cible PRD ~9.4%)
+--------------------+-----+------+------+--------+------------+-------+--------+--------+--------+---+----+---------+-----------------+
|HeartDiseaseorAttack|  BMI|Smoker|Stroke|Diabetes|PhysActivity|GenHlth|MentHlth|PhysHlth|DiffWalk|Sex| Age|Education|HvyAlcoholConsump|
+--------------------+-----+------+------+--------+------------+-------+--------+--------+--------+---+----+---------+-----------------+
|                 0.0| 16.6|   1.0|   0.0|     1.0|         1.0|    2.0|    30.0|     3.0|     0.0|1.0| 8.0|      6.0|             NULL|
|                 0.0|29.18|  NULL|   0.0|     0.0|         1.0|  

In [151]:
# Sauvegarder le DataFrame brut
output_path = "/kaggle/working/brfss_selected.parquet"
df_spark.write.mode("overwrite").parquet(output_path)
print(f"Donnees sauvegardees : {output_path}")

Donnees sauvegardees : /kaggle/working/brfss_selected.parquet


## Resume du notebook 01

- Dataset BRFSS telecharge depuis Kaggle
- Session Spark initialisee en mode local
- Donnees chargees et inspectees
- 18 variables selectionnees
- DataFrame sauvegarde en Parquet

**-> Passer au notebook 02 pour le preprocessing et l'EDA**